# 05 - Test Report Agent (Gemma 4) di Google Colab

Notebook ini **khusus untuk dijalankan di Google Colab dengan GPU** — beda
dari notebook lain di folder ini yang jalan di CPU biasa. Tujuannya
mencoba `utils/report_agent.py::generate_report()` (narasi "Alasan" dari
LLM Gemma 4) secara langsung, **tanpa lewat UI Streamlit** — cara paling
cepat untuk lihat kualitas narasinya & berapa lama generate-nya sebelum
dites lewat halaman "Pengajuan Credit Baru" yang sesungguhnya.

> ⚠️ **Notebook ini belum pernah dieksekusi** — dibuat & ditulis di
> environment tanpa GPU (tidak bisa download/jalankan Gemma 4 di sana),
> jadi tidak ada output tersimpan di sel manapun. Jalankan dari atas ke
> bawah di Colab.

## Sebelum mulai

1. **Ganti runtime ke GPU**: menu `Runtime` → `Change runtime type` →
   pilih `T4 GPU` (gratis) atau lebih tinggi kalau tersedia.
2. **Akses model Gemma 4**: buka
   [huggingface.co/google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it),
   login, dan setujui lisensi model (biasanya perlu klik "Agree and access
   repository" sekali). Siapkan HuggingFace access token
   (Settings → Access Tokens) untuk login di Sel 3 di bawah.
3. Pastikan perubahan terbaru project sudah ke-push ke GitHub (`git push`)
   sebelum menjalankan notebook ini — sel clone di bawah mengambil dari
   `origin/main`.

In [ ]:
!nvidia-smi

## 1. Clone Repo & Install Dependencies

`torch`/`transformers`/`accelerate` sengaja **tidak** ada di
`requirements.txt` project (itu khusus untuk deploy Streamlit Cloud yang
tidak punya GPU) — jadi diinstall terpisah di sini.

In [ ]:
REPO_URL = "https://github.com/indahsyafhyra12/Capstone-Project-ODP-DA-Asek.git"
REPO_DIR = "Capstone-Project-ODP-DA-Asek"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate huggingface_hub bitsandbytes

## 2. Login HuggingFace (wajib untuk model gated seperti Gemma)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Import Modul Project

Dijalankan dari root repo (`%cd` di atas), jadi `utils.*` bisa langsung
di-import tanpa perlu `sys.path` tambahan.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd

from utils.feature_builder import load_raw_tables
from utils.risk_ml_pipeline import predict_credit_screening
from utils.report_agent import generate_report, generate_reports_batch, MODEL_ID

print(f"Model target: {MODEL_ID}")

master_dataset = pd.read_csv("data/processed/master_dataset.csv", dtype={"NIK": str})
master_scored = pd.read_csv("data/processed/master_scored.csv", dtype={"NIK": str})
print(f"master_dataset: {master_dataset.shape}, master_scored: {master_scored.shape}")

## 4. Preload Model (Sekali Saja)

Panggilan pertama ke `generate_report()` yang men-trigger download +
load model ke GPU (bisa beberapa menit tergantung koneksi) — dijalankan
di sel terpisah supaya waktu download tidak tercampur ke timing generate
per-nasabah di sel berikutnya.

In [ ]:
_warm_start = time.perf_counter()
from utils.report_agent import _load_model
_processor, _model = _load_model()
print(f"Model dimuat dalam {time.perf_counter() - _warm_start:.1f}s")
print(f"Device: {_model.device}")

## 5. Test 5 Kasus Representatif (sama seperti notebook 04)

Mencakup: Layak, Layak Bersyarat, Perlu Review Ulang, Tidak Layak (lewat
ML), dan Tidak Layak (hard-reject DHN — STAGE 1, tidak pernah sampai ML/
LLM sama sekali, buat lihat narasinya tetap masuk akal utk kasus ini).

In [ ]:
TEST_IDS = {
    "APP202600001": "Layak",
    "APP202600008": "Layak Bersyarat",
    "APP202600003": "Perlu Review Ulang",
    "APP202600576": "Tidak Layak (skor rendah, lewat ML)",
    "APP202600020": "Tidak Layak (hard-reject DHN, STAGE 1)",
}

results = []
for app_id, label in TEST_IDS.items():
    row = master_dataset[master_dataset["application_id"] == app_id].iloc[0]
    company_name = row["company_name"]
    pred = predict_credit_screening(row.to_dict())
    prompt_row = {"company_name": company_name, **pred}

    start = time.perf_counter()
    narrative = generate_report(prompt_row)
    elapsed = time.perf_counter() - start

    is_fallback = narrative == pred["insight"]
    results.append({
        "application_id": app_id, "expected": label, "company_name": company_name,
        "decision": pred["decision"], "risk_score": pred["risk_score"],
        "elapsed_sec": round(elapsed, 2), "is_fallback": is_fallback,
        "insight_rule_based": pred["insight"], "narrative_llm": narrative,
    })

    print("=" * 80)
    print(f"{app_id} ({company_name}) — ekspektasi: {label}")
    print(f"Decision: {pred['decision']} | risk_score: {pred['risk_score']} | waktu generate: {elapsed:.2f}s")
    print(f"Fallback ke rule-based? {'YA - cek log warning di atas' if is_fallback else 'TIDAK - narasi LLM asli'}")
    print()
    print("[Insight rule-based]")
    print(pred["insight"])
    print()
    print("[Narasi LLM]")
    print(narrative)
print("=" * 80)

In [ ]:
summary_df = pd.DataFrame(results)[["application_id", "expected", "decision", "risk_score", "elapsed_sec", "is_fallback"]]
summary_df

## 6. Cek Konsistensi Guardrail Manual

Baca ulang tiap narasi di atas dan cek manual: apakah kata-katanya tidak
bertentangan dengan `decision`-nya (mis. narasi utk "Tidak Layak" tidak
menyiratkan disetujui, dan sebaliknya)? Kolom `is_fallback=True` di tabel
atas berarti guardrail otomatis (`_sanity_check`) sudah mendeteksi masalah
duluan dan fallback ke rule-based — kalau itu terjadi cukup sering, cek log
`WARNING` di output Sel 6 untuk lihat narasi asli yang ditolak.

## 7. Test Batch (`generate_reports_batch`)

Sample kecil (10 baris, hanya yang lolos hard-rule / `risk_score` terisi)
dari `master_scored.csv` — simulasi pemakaian notebook untuk mengisi ulang
kolom insight versi LLM secara massal.

In [ ]:
sample = master_scored[master_scored["risk_score"].notna()].sample(10, random_state=42)

batch_start = time.perf_counter()
sample_narratives = generate_reports_batch(sample)
batch_elapsed = time.perf_counter() - batch_start

print(f"Total waktu utk {len(sample)} baris: {batch_elapsed:.1f}s (rata-rata {batch_elapsed/len(sample):.2f}s/baris)")

batch_preview = sample[["application_id", "company_name", "decision", "risk_score"]].copy()
batch_preview["narrative_llm"] = sample_narratives.values
batch_preview

## 8. Evaluasi Otomatis dengan DeepEval

Bagian 6 di atas ("Cek Konsistensi Guardrail Manual") minta narasinya dibaca
satu-satu secara manual. Bagian ini menambahkan lapisan evaluasi **otomatis
& terukur** pakai [DeepEval](https://github.com/confident-ai/deepeval)
supaya kualitas narasi bisa dicek konsisten tiap kali `SYSTEM_PROMPT`/model
diganti, tanpa baca ulang manual satu-per-satu.

Prinsip yang sama dengan desain project ini (lihat docstring
`utils/report_agent.py`): **semua lokal, tanpa API key**. Semua metric
DeepEval (GEval, Faithfulness, dst.) butuh LLM sebagai "judge" penilai, dan
default DeepEval adalah OpenAI GPT-4o (butuh API key + internet) — bagian
ini pakai **judge lokal open-source**, sengaja beda model dari Gemma
(generator) supaya model tidak menilai keluarannya sendiri (self-grading
bias).

Judge yang dipakai: `Qwen/Qwen2.5-7B-Instruct` (lisensi Apache-2.0, tidak
digated seperti Gemma — tidak perlu approve lisensi/login HF terpisah),
di-quantize 4-bit NF4 sama seperti pola `_load_model()` di
`report_agent.py`/`kwitansi_extractor.py`. Kalau kena OOM di GPU T4 gratis
(Gemma + Qwen 7B sekaligus dalam 4-bit biasanya muat di 16GB, tapi mepet
kalau ditambah activation/generation cache), ada sel opsional di bawah
untuk melepas Gemma dari GPU dulu sebelum load judge, atau ganti
`JUDGE_MODEL_ID` ke varian lebih kecil (mis. `Qwen/Qwen2.5-3B-Instruct`).

> Bagian ini juga **belum pernah dieksekusi** (sama seperti bagian atas) —
> jalankan dari Bagian 1 s/d sini secara berurutan di Colab/Kaggle GPU.

In [ ]:
!pip install -q -U deepeval

import deepeval
print(f"deepeval version: {deepeval.__version__}")

### 8.1 Kumpulkan Narasi untuk Dievaluasi (pakai Gemma yang sudah dimuat)

Digabung dari 2 sumber:
- 5 kasus representatif Bagian 5 (`results`) — narasinya dipakai ulang,
  TIDAK di-generate ulang (sudah termasuk 1 kasus hard-reject DHN, STAGE 1).
- Sample tambahan stratifikasi per `insight_kategori` dari
  `master_scored.csv` (`N_PER_KATEGORI` baris/kategori) — supaya tiap
  kategori narasi (Layak karena/tapi, Layak bersyarat karena, Perlu review
  ulang karena, Tidak layak karena) punya lebih dari 1 contoh.

Dikumpulkan sebagai `eval_records` (list of dict) dulu, BUKAN langsung jadi
`LLMTestCase` — supaya proses generate narasi (butuh Gemma) selesai duluan
sebelum GPU dibebani judge tambahan di 8.2.

In [ ]:
from utils.report_agent import _build_prompt_data, USER_PROMPT_TEMPLATE


def build_input_prompt(row: dict) -> str:
    return USER_PROMPT_TEMPLATE.format(**_build_prompt_data(row))


def build_retrieval_context(row: dict) -> list:
    d = _build_prompt_data(row)
    return [
        f"Keputusan: {d['decision']} (Zona: {d['zone']})",
        f"Skor gabungan: {d['risk_score']}",
        f"Alasan inti sistem (WAJIB dipertahankan maknanya): {d['insight']}",
        f"Character/Credit History: {d['character_notes']} (skor {d['character_score']})",
        f"Collateral: {d['collateral_notes']} (skor {d['collateral_score']})",
        f"Financial: {d['financial_notes']} (skor {d['financial_score']})",
        f"Cashflow: {d['cashflow_notes']} (skor {d['cashflow_score']})",
        f"Rekomendasi: {d['jenis_kredit_rekomendasi']}, nominal Rp{d['nominal_disetujui']:,}, "
        f"tenor {d['jangka_waktu_bulan']} bulan, bunga {d['bunga_persen']}%",
    ]


eval_records = []

# (a) 5 kasus representatif dari Bagian 5 - reuse narasi, jangan generate ulang
for r in results:
    row_full = master_dataset[master_dataset["application_id"] == r["application_id"]].iloc[0]
    pred_full = predict_credit_screening(row_full.to_dict())
    prompt_row_full = {"company_name": row_full["company_name"], **pred_full}
    eval_records.append({
        "label": f"{r['application_id']} ({r['expected']})",
        "input": build_input_prompt(prompt_row_full),
        "actual_output": r["narrative_llm"],
        "retrieval_context": build_retrieval_context(prompt_row_full),
    })

# (b) sample tambahan stratifikasi per kategori dari master_scored.csv
N_PER_KATEGORI = 2
extra_sample = (
    master_scored[master_scored["risk_score"].notna()]
    .groupby("insight_kategori", group_keys=False)
    .apply(lambda g: g.sample(min(N_PER_KATEGORI, len(g)), random_state=42))
)
print(f"Sample tambahan: {len(extra_sample)} baris dari {extra_sample['insight_kategori'].nunique()} kategori")

for _, row in extra_sample.iterrows():
    row_dict = row.to_dict()
    narrative = generate_report(row_dict)
    eval_records.append({
        "label": f"{row_dict['application_id']} ({row_dict['insight_kategori']})",
        "input": build_input_prompt(row_dict),
        "actual_output": narrative,
        "retrieval_context": build_retrieval_context(row_dict),
    })

print(f"Total kasus untuk dievaluasi: {len(eval_records)}")

### (Opsional) Lepas Gemma dari GPU Sebelum Load Judge

Jalankan sel ini **hanya kalau** sel load judge di Bagian 8.2 kena OOM.
Setelah ini, sel-sel Bagian 5-7 di atas (yang panggil `generate_report()`)
tidak bisa dijalankan ulang tanpa reload Gemma lagi.

In [ ]:
# Jalankan HANYA kalau load judge di 8.2 kena OOM.
import gc
import torch
from utils.report_agent import _load_model as _load_report_model

_load_report_model.clear()  # kosongkan cache st.cache_resource
del _model, _processor
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory allocated setelah Gemma dilepas: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 8.2 Load Judge LLM (terpisah dari Gemma)

`DeepEvalBaseLLM` custom (API dikonfirmasi dari source `deepeval==4.2.0`:
`load_model()`, `generate()`, `a_generate()`, `get_model_name()`) — pola
loading 4-bit NF4-nya sama seperti `_load_model()` di
`report_agent.py`/`kwitansi_extractor.py`, tapi TIDAK pakai
`@st.cache_resource` karena notebook ini jalan lepas dari Streamlit.

In [ ]:
from deepeval.models import DeepEvalBaseLLM

JUDGE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # ganti ke -3B-Instruct kalau OOM


class LocalJudgeLLM(DeepEvalBaseLLM):
    def __init__(self, model_id: str = JUDGE_MODEL_ID):
        self.model_id = model_id
        super().__init__(model=model_id)

    def load_model(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        model = AutoModelForCausalLM.from_pretrained(
            self.model_id, quantization_config=quant_config, device_map="auto",
        )
        return model

    def generate(self, prompt: str, schema=None) -> str:
        import torch

        # `return_dict=True` dipaksa eksplisit (sama seperti pola
        # _run_generation() di utils/report_agent.py) - beberapa versi
        # transformers mengembalikan BatchEncoding (bukan Tensor polos) dari
        # apply_chat_template walau return_dict tidak diminta, yang bikin
        # `.shape` di bawah gagal (AttributeError) kalau diasumsikan Tensor.
        messages = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True,
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[-1]
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, max_new_tokens=768, do_sample=False)
        text = self.tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True)
        return text.strip()

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self.generate(prompt, schema=schema)

    def get_model_name(self) -> str:
        return self.model_id


_judge_start = time.perf_counter()
judge = LocalJudgeLLM()
print(f"Judge ({JUDGE_MODEL_ID}) dimuat dalam {time.perf_counter() - _judge_start:.1f}s")
print(f"Device: {judge.model.device}")

### 8.3 Definisikan Metric

- **Decision Consistency** (`GEval` custom criteria) — otomatisasi dari
  aturan `_sanity_check()` yang sudah ada di `report_agent.py`, tapi lebih
  ketat: bukan cuma cek kata kunci positif/negatif, judge LLM menilai apakah
  narasi BENAR-BENAR konsisten dengan Keputusan/Zona di `input`, termasuk
  aturan khusus hard-rule (poin 5 di `SYSTEM_PROMPT`).
- **Faithfulness** (metric bawaan DeepEval) — cek narasi tidak mengandung
  klaim yang tidak didukung oleh `retrieval_context` (poin 2 di
  `SYSTEM_PROMPT`: "JANGAN mengarang angka/fakta").

`async_mode=False` di kedua metric supaya jalan sinkron sederhana di
notebook (`judge` cuma pegang 1 GPU, tidak ada gunanya paralel semu).

In [ ]:
from deepeval.metrics import GEval, FaithfulnessMetric
from deepeval.test_case import SingleTurnParams

decision_consistency = GEval(
    name="Decision Consistency",
    criteria=(
        "Data hasil screening (skor, Keputusan, Zona, alasan tiap komponen) ada di 'input'. "
        "Narasi di 'actual_output' HARUS konsisten dengan Keputusan & Zona di 'input' - TIDAK "
        "boleh menyiratkan hasil yang bertentangan (mis. menyarankan disetujui padahal "
        "Keputusan=Tidak Layak, atau menyiratkan ditolak/perlu perbaikan besar padahal "
        "Keputusan=Layak). Kalau 'input' menyebut kondisi hard-rule (NIK tidak valid, DHN, "
        "SLIK Macet), narasi WAJIB menyampaikan penolakan sebagai OTOMATIS dan MUTLAK sesuai "
        "kebijakan bank, tanpa menyiratkan ada ruang negosiasi."
    ),
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=judge,
    threshold=0.7,
    async_mode=False,
)

faithfulness = FaithfulnessMetric(
    threshold=0.7, model=judge, include_reason=True, async_mode=False,
)

### 8.4 Jalankan Evaluasi

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.evaluate import evaluate, AsyncConfig, DisplayConfig

eval_test_cases = [
    LLMTestCase(
        input=rec["input"], actual_output=rec["actual_output"],
        context=rec["retrieval_context"], retrieval_context=rec["retrieval_context"],
    )
    for rec in eval_records
]

eval_result = evaluate(
    test_cases=eval_test_cases,
    metrics=[decision_consistency, faithfulness],
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(show_indicator=True, print_results=True, inspect_after_run=False),
)

In [ ]:
summary_rows = []
for rec, tr in zip(eval_records, eval_result.test_results):
    for md in tr.metrics_data:
        summary_rows.append({
            "kasus": rec["label"], "metric": md.name,
            "score": round(md.score, 3) if md.score is not None else None,
            "pass": md.success, "reason": md.reason,
        })

eval_summary_df = pd.DataFrame(summary_rows)
print(f"Pass rate keseluruhan: {eval_summary_df['pass'].mean() * 100:.1f}% "
      f"({int(eval_summary_df['pass'].sum())}/{len(eval_summary_df)} metric-case lolos threshold)")
print()
print(eval_summary_df.groupby("metric")["pass"].mean().rename("pass_rate"))
eval_summary_df

### Cara Baca Hasil

- Baris `pass=False` di `eval_summary_df` = kasus yang perlu dibaca manual
  kolom `reason`-nya (alasan judge LLM menilai gagal) — biasanya menandakan
  `SYSTEM_PROMPT` perlu contoh tambahan untuk kasus itu, atau
  `_sanity_check()` butuh disesuaikan.
- Kalau `pass_rate` metric **Decision Consistency** rendah tapi baca manual
  narasinya sebenarnya oke → kemungkinan judge (`Qwen2.5-7B-Instruct`)
  terlalu ketat/salah paham kriteria — coba ganti `JUDGE_MODEL_ID` ke model
  lain atau perhalus kalimat `criteria` di Sel 8.3.
- Kalau `reason` banyak berisi pesan error parsing (bukan alasan substantif)
  → tanda judge lokal gagal ikuti format output yang diminta DeepEval —
  model judge kekecilan/kurang instruction-following, pertimbangkan model
  judge yang lebih besar kalau resource GPU cukup.
- Skor di sini melengkapi (BUKAN mengganti) guardrail otomatis
  `_sanity_check()` yang tetap jalan di production (`generate_report()`) —
  evaluasi ini untuk validasi kualitas SEBELUM deploy/ganti prompt, bukan
  pengganti guardrail runtime.

## Ringkasan & Langkah Selanjutnya

- Kalau narasi di Bagian 5 sudah bagus & `is_fallback` jarang `True`,
  `generate_report()` siap dipakai — tidak perlu ubah apa pun di
  `utils/report_agent.py`.
- Kalau `is_fallback` sering `True`, buka log `WARNING` di atas untuk
  lihat narasi asli yang ditolak guardrail — mungkin perlu sesuaikan
  `_sanity_check()` (terlalu ketat) atau `SYSTEM_PROMPT` (model sering
  keluar dari format).
- Waktu generate per-nasabah di kolom `elapsed_sec` (Bagian 5) menentukan
  perlu tidaknya UX tambahan (progress indicator, dsb) di halaman
  "Pengajuan Credit Baru" — saat ini sudah ada `st.spinner`, cek apakah
  itu cukup atau perlu progress bar kalau ternyata generate-nya lama.
- Untuk test UI Streamlit-nya juga (bukan cuma fungsi `generate_report()`
  langsung), jalankan `streamlit run app.py` di Colab lalu expose lewat
  `pyngrok`/`localtunnel` — beri tahu kalau mau notebook terpisah utk itu.
- Bagian 8 (DeepEval) menggantikan pembacaan manual di Bagian 6 dengan skor
  terukur (`eval_summary_df`) — jalankan itu tiap kali `SYSTEM_PROMPT` atau
  `MODEL_ID` di `report_agent.py` diganti, supaya ada baseline pass rate
  untuk dibandingkan sebelum & sesudah perubahan.